In [1]:
pip install numpy matplotlib scipy ipykernel



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:

import sys
sys.path.insert(0, "../src")
import jssp_fuzzy_ga.core as core_mod

print("Loaded from:", core_mod.__file__)
print("Has make_instance:", hasattr(core_mod, "make_instance"))

Loaded from: /Users/krrishts/Documents/Fuzzy-mutation-GA/fuzzy-ga/notebooks/../src/jssp_fuzzy_ga/core.py
Has make_instance: True


In [3]:
import numpy as np
from jssp_fuzzy_ga.core import make_instance, FT06_OPTIMUM

inst = make_instance("ft06")
rng = np.random.default_rng(0)
chrom = inst.random_chromosome(rng)

print("chromosome length:", len(chrom), "| expected:", inst.chrom_len)
print("chromosome:", chrom)


chromosome length: 36 | expected: 36
chromosome: [0 5 5 0 0 3 4 3 1 0 0 3 4 1 1 1 5 3 1 2 3 2 5 2 1 0 2 4 2 4 4 3 4 5 2 5]


In [4]:
print("n_jobs:", inst.n_jobs)
print("n_ops_per_job:", inst.n_ops_per_job)
print("chrom_len:", inst.chrom_len)
print("n_machines:", inst.n_machines)

n_jobs: 6
n_ops_per_job: [6, 6, 6, 6, 6, 6]
chrom_len: 36
n_machines: 6


In [5]:
genes_before = []
for j, n_ops in enumerate(inst.n_ops_per_job):
    genes_before += [j] * n_ops
print("before shuffle:", genes_before)

rng2 = np.random.default_rng(0)
chrom2 = inst.random_chromosome(rng2)
print("after shuffle: ", chrom2.tolist())

before shuffle: [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5]
after shuffle:  [0, 5, 5, 0, 0, 3, 4, 3, 1, 0, 0, 3, 4, 1, 1, 1, 5, 3, 1, 2, 3, 2, 5, 2, 1, 0, 2, 4, 2, 4, 4, 3, 4, 5, 2, 5]


In [6]:
def decode_verbose(inst, chromosome, max_steps=12):
    next_op = [0] * inst.n_jobs
    machine_free = [0] * inst.n_machines
    job_free = [0] * inst.n_jobs
    for step, gene in enumerate(chromosome):
        j = int(gene)
        op_idx = next_op[j]
        m, dur = inst.jobs[j][op_idx]
        start = max(machine_free[m], job_free[j])
        finish = start + dur
        machine_free[m] = finish
        job_free[j] = finish
        next_op[j] += 1
        if step < max_steps:
            print(f"step {step:2d}: job {j} op {op_idx} -> machine {m} (dur {dur}) | starts {start}, finishes {finish}")
    return max(job_free)

makespan = decode_verbose(inst, chrom)
print("\nfinal makespan:", makespan)

step  0: job 0 op 0 -> machine 2 (dur 1) | starts 0, finishes 1
step  1: job 5 op 0 -> machine 1 (dur 3) | starts 0, finishes 3
step  2: job 5 op 1 -> machine 3 (dur 3) | starts 3, finishes 6
step  3: job 0 op 1 -> machine 0 (dur 3) | starts 1, finishes 4
step  4: job 0 op 2 -> machine 1 (dur 6) | starts 4, finishes 10
step  5: job 3 op 0 -> machine 1 (dur 5) | starts 10, finishes 15
step  6: job 4 op 0 -> machine 2 (dur 9) | starts 1, finishes 10
step  7: job 3 op 1 -> machine 0 (dur 5) | starts 15, finishes 20
step  8: job 1 op 0 -> machine 1 (dur 8) | starts 15, finishes 23
step  9: job 0 op 3 -> machine 3 (dur 7) | starts 10, finishes 17
step 10: job 0 op 4 -> machine 5 (dur 3) | starts 17, finishes 20
step 11: job 3 op 2 -> machine 2 (dur 5) | starts 20, finishes 25

final makespan: 87


In [7]:
from jssp_fuzzy_ga.core import tournament_select, pox_crossover, swap_mutation, population_diversity


In [8]:
# build a tiny fake population + fitness to test selection in isolation
pop_test = [inst.random_chromosome(np.random.default_rng(i)) for i in range(5)]
fitness_test = [inst.decode(c) for c in pop_test]
print("fitnesses:", fitness_test)

winner = tournament_select(pop_test, fitness_test, np.random.default_rng(0), k=3)
print("winner makespan:", inst.decode(winner))

fitnesses: [87, 98, 85, 82, 89]
winner makespan: 82


In [9]:
rng_demo = np.random.default_rng(0)
idx_shown = rng_demo.integers(0, len(pop_test), size=3)
print("indices drawn into this tournament:", idx_shown)
print("their fitnesses:", [fitness_test[i] for i in idx_shown])

indices drawn into this tournament: [4 3 2]
their fitnesses: [89, 82, 85]


In [10]:
p1 = inst.random_chromosome(np.random.default_rng(1))
p2 = inst.random_chromosome(np.random.default_rng(2))
child = pox_crossover(p1, p2, inst.n_jobs, np.random.default_rng(0))

print("parent1:", p1)
print("parent2:", p2)
print("child:  ", child)

import collections
print("child gene counts:", collections.Counter(child.tolist()))

parent1: [1 0 1 1 5 0 4 2 2 3 2 0 1 4 3 3 5 4 4 5 5 2 3 5 4 0 4 2 1 3 0 0 1 5 3 2]
parent2: [4 5 2 1 0 3 1 4 4 2 4 1 2 3 2 4 5 3 5 0 3 1 2 0 0 2 3 1 0 3 5 4 5 5 1 0]
child:   [4 5 2 1 0 1 4 4 2 3 4 1 2 2 3 3 4 5 5 0 1 2 3 0 0 2 1 0 5 3 4 5 5 1 3 0]
child gene counts: Counter({4: 6, 5: 6, 2: 6, 1: 6, 0: 6, 3: 6})


In [11]:
n_jobs = inst.n_jobs
jobs_shuffled = np.arange(n_jobs)
rng_demo = np.random.default_rng(0)
rng_demo.shuffle(jobs_shuffled)
split = rng_demo.integers(1, n_jobs)
j1 = set(jobs_shuffled[:split].tolist())

print("shuffled job order:", jobs_shuffled)
print("split point:", split)
print("jobs taken from parent1 (j1):", j1)
print("jobs taken from parent2 (the rest):", set(range(n_jobs)) - j1)

shuffled job order: [3 2 5 4 0 1]
split point: 1
jobs taken from parent1 (j1): {3}
jobs taken from parent2 (the rest): {0, 1, 2, 4, 5}


In [12]:
job_to_check = 3  # the only job taken from parent1

# find every position where job_to_check appears in the child
child_positions = [i for i, g in enumerate(child) if g == job_to_check]
p1_positions = [i for i, g in enumerate(p1) if g == job_to_check]

print(f"positions of job {job_to_check} in child:  ", child_positions)
print(f"positions of job {job_to_check} in parent1:", p1_positions)
print("identical positions?", child_positions == p1_positions)

positions of job 3 in child:   [9, 14, 15, 22, 29, 34]
positions of job 3 in parent1: [9, 14, 15, 22, 29, 34]
identical positions? True


In [13]:
job_to_check2 = 0  # any job NOT in j1

child_seq = [g for g in child if g == job_to_check2]
p2_seq = [g for g in p2 if g == job_to_check2]
print(f"child's job-{job_to_check2} genes, in order: {child_seq}")
print(f"parent2's job-{job_to_check2} genes, in order: {p2_seq}")
# these will trivially match since they're all just job_to_check2 repeated --
# the REAL check is whether the relative ORDER of job {job_to_check2}'s
# operations vs other non-j1 jobs matches parent2's order:
child_non_j1 = [g for g in child if g != job_to_check][:15]
p2_non_j1 = [g for g in p2 if g != job_to_check][:15]
print("\nfirst 15 non-job-3 genes in child: ", child_non_j1)
print("first 15 non-job-3 genes in parent2:", p2_non_j1)
print("match?", child_non_j1 == p2_non_j1)

child's job-0 genes, in order: [0, 0, 0, 0, 0, 0]
parent2's job-0 genes, in order: [0, 0, 0, 0, 0, 0]

first 15 non-job-3 genes in child:  [4, 5, 2, 1, 0, 1, 4, 4, 2, 4, 1, 2, 2, 4, 5]
first 15 non-job-3 genes in parent2: [4, 5, 2, 1, 0, 1, 4, 4, 2, 4, 1, 2, 2, 4, 5]
match? True


In [14]:
rng_mut = np.random.default_rng(0)
original = inst.random_chromosome(np.random.default_rng(5))
mutated = swap_mutation(original, rate=1.0, rng=rng_mut)  # rate=1.0 forces a swap every time

print("original:", original)
print("mutated: ", mutated)
print("same gene counts:", collections.Counter(original.tolist()) == collections.Counter(mutated.tolist()))

diff_positions = [i for i in range(len(original)) if original[i] != mutated[i]]
print("positions that changed:", diff_positions)

original: [2 1 0 1 5 4 2 4 1 3 5 5 1 3 5 5 3 4 0 4 1 4 2 4 0 2 5 3 3 2 3 0 0 2 1 0]
mutated:  [2 1 0 1 5 4 2 4 1 0 5 5 1 3 5 5 3 4 3 4 1 4 2 4 0 2 5 3 3 2 3 0 0 2 1 0]
same gene counts: True
positions that changed: [9, 18]
